In [0]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

# 1. Read from the Bronze Table
bronze_df = spark.read.table("f1_project.bronze.raw_sessions")

# 2. Transform the Data
silver_df = (bronze_df
    # Convert strings to Timestamps (ISO 8601 format)
    .withColumn("start_time", to_timestamp(col("date_start")))
    .withColumn("end_time", to_timestamp(col("date_end")))
    
    # Filter for meaningful sessions (Race/Qualifying)
    .filter(col("session_type").isin("Race", "Qualifying"))
    
    # Select and Rename columns to be more 'Professional'
    .select(
        col("session_key").alias("session_id"),
        col("session_name"),
        col("session_type"),
        col("location"),
        col("country_name"),
        "start_time",
        "end_time",
        col("year").cast("int")
    )
    # Add an audit column
    .withColumn("processed_at", current_timestamp())
)

# 3. Save to Silver Table (Using Delta format)
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("f1_project.silver.cleaned_sessions"))

print("Silver Transformation Complete!")
display(spark.table("f1_project.silver.cleaned_sessions"))